In [1]:
import sys
import argparse

import os
import scripts.utils_forTraining as utils
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from EPInformer.models import EPInformer_v2, enhancer_predictor_256bp
from scipy import stats
from tqdm import tqdm
import torch
from torch.utils.data import Subset, Dataset
from sklearn.model_selection import GroupKFold

In [2]:
def transform_data(transform, data):
    if transform == 'log10':
        return np.log10(data + 1)
    elif transform == 'tanh':
        return np.tanh(data)
    elif transform == 'sigmoid':
        return 1 / (1 + np.exp(-data))

# RNA Embedding

In [3]:
cell = 'K562'
expr_type = 'CAGE'
n_extraFeat = 3
n_enhancers = 60
hic_threshold = None
distance_threshold = 100000
rna_encoding = False
rna_embedding = True
rna_transform = 'log10'

#################
EP_df = pd.read_csv(f'./data/{cell}_enhancer_gene_links_100kb.hg38.tsv', sep='\t')
promoter_df = EP_df.groupby('TargetGeneEnsembl_ID', as_index = False)['chr'].first()
promoter_df.rename(columns={'TargetGeneEnsembl_ID': 'Ensembl_ID'}, inplace=True)
all_ds = utils.promoter_enhancer_dataset(data_folder= './data/', expr_type=expr_type, cell_type=cell, 
                                         n_extraFeat=n_extraFeat, usePromoterSignal=True, n_enhancers=n_enhancers, 
                                         hic_threshold=hic_threshold, distance_threshold=distance_threshold, 
                                         rna_encoding=rna_encoding, rna_embedding=rna_embedding, rna_transform=rna_transform)
ensid_list = [eid.decode() for eid in all_ds.data_h5['ensid'][:]]
ensid_df = pd.DataFrame(ensid_list, columns=['ensid'])
ensid_df['idx'] = np.arange(len(ensid_list))
ensid_df = ensid_df.set_index('ensid')

In [5]:
input_PE, input_feat, input_dist, y_expr, eid, rna_embedding = all_ds[0]
rna_embedding

tensor([[0.1143, 0.1130, 0.1026,  ..., 0.4431, 0.5873, 0.6026],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])

In [ ]:
input_PE, input_feat, input_dist, y_expr, eid, rna = all_ds[0]
rna

In [ ]:
rna_embedding = all_ds.data_h5['rna'][0]
rna_embedding

In [ ]:
transform_data('log10', rna_embedding).shape

In [ ]:
plt.clf()
plt.hist(rna_embedding.flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('log10', rna_embedding).flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('tanh', rna_embedding).flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('sigmoid', rna_embedding).flatten(), bins=30)
plt.show()

# RNA Encoding

In [ ]:
cell = 'GM12878'
expr_type = 'CAGE'
n_extraFeat = 3
n_enhancers = 60
hic_threshold = None
distance_threshold = 100000
rna_encoding = True
rna_embedding = False
rna_transform = 'sigmoid'

#################
EP_df = pd.read_csv(f'./data/{cell}_enhancer_gene_links_100kb.hg38.tsv', sep='\t')
promoter_df = EP_df.groupby('TargetGeneEnsembl_ID', as_index = False)['chr'].first()
promoter_df.rename(columns={'TargetGeneEnsembl_ID': 'Ensembl_ID'}, inplace=True)
all_ds = utils.promoter_enhancer_dataset(data_folder= './data/', expr_type=expr_type, cell_type=cell, 
                                         n_extraFeat=n_extraFeat, usePromoterSignal=True, n_enhancers=n_enhancers, 
                                         hic_threshold=hic_threshold, distance_threshold=distance_threshold, 
                                         rna_encoding=rna_encoding, rna_embedding=rna_embedding, rna_transform=rna_transform)
ensid_list = [eid.decode() for eid in all_ds.data_h5['ensid'][:]]
ensid_df = pd.DataFrame(ensid_list, columns=['ensid'])
ensid_df['idx'] = np.arange(len(ensid_list))
ensid_df = ensid_df.set_index('ensid')

In [ ]:
input_PE, input_feat, input_dist, y_expr, eid = all_ds[0]
input_PE

In [ ]:
signal = seq_code[:, :, 4]
signal

In [ ]:
signal.shape

In [ ]:
#promoter_code = seq_code[:, 0]
#promoter_code

In [ ]:
#signal = promoter_code[:, :, 4].flatten()
#signal

In [ ]:
plt.clf()
plt.hist(signal.flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('log10', signal).flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('tanh', signal).flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('sigmoid', signal).flatten(), bins=30)
plt.show()

In [ ]:
promoter_signal = signal[:, 0, :]
promoter_signal

In [ ]:
promoter_signal[-1, 0:1000]

In [ ]:
plt.clf()
plt.hist(promoter_signal.flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('log10', promoter_signal).flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('tanh', promoter_signal).flatten(), bins=30)
plt.show()

In [ ]:
plt.clf()
plt.hist(transform_data('sigmoid', promoter_signal).flatten(), bins=30)
plt.show()